#  GroupDNA


### Submitted By:
**shivang jani**




In this project we are analyze a WhatsApp chat dataset using Python fundamentals and NumPy. The program extracts useful information from the chat and generates insights such as participant activity, message statistics, busiest hours, and personality analysis.

## Read the Dataset

In this step, we open the WhatsApp chat file and read all the lines for further processing.

In [2]:
import numpy as np
file = open("hostel_bois.txt", "r", encoding="utf-8")
lines = file.readlines()
file.close()
print("Dataset loaded successfully.")
print("Total lines:", len(lines))

Dataset loaded successfully.
Total lines: 3178


Feature 1 – Chat Parser

This feature reads the exported WhatsApp chat file and processes it line by line. Each valid message is separated into date, time, sender name, and message text. The extracted information is stored in a list of dictionaries for further analysis. System messages are identified and counted separately.

In [3]:
messages = []
system_messages = 0


for line in lines:


    line = line.strip()


    if line == "":
        continue


    parts = line.split(" - ", 1)


    if len(parts) < 2:
        continue

    date_time = parts[0]
    remaining = parts[1]


    if ": " not in remaining:
        system_messages = system_messages + 1
        continue


    date = date_time.split(", ")[0]
    time = date_time.split(", ")[1]


    sender = remaining.split(": ", 1)[0]
    message = remaining.split(": ", 1)[1]


    data = {
        "date": date,
        "time": time,
        "sender": sender,
        "message": message
    }


    messages.append(data)


print("Feature 1 Completed Successfully")
print("--------------------------------")
print("Total Chat Messages :", len(messages))
print("System Messages :", system_messages)

print("\nFirst 5 Messages\n")

for i in range(5):
    print(messages[i])

Feature 1 Completed Successfully
--------------------------------
Total Chat Messages : 3174
System Messages : 4

First 5 Messages

{'date': '01/04/24', 'time': '01:17', 'sender': 'Rahul', 'message': 'scene fix'}
{'date': '01/04/24', 'time': '01:17', 'sender': 'Rahul', 'message': 'haan'}
{'date': '01/04/24', 'time': '01:18', 'sender': 'Rahul', 'message': 'kya scene'}
{'date': '01/04/24', 'time': '02:13', 'sender': 'Rahul', 'message': 'abhi free hai?'}
{'date': '01/04/24', 'time': '02:13', 'sender': 'Rahul', 'message': 'abey'}


Feature 2 – Group Overview

This feature provides an overall summary of the WhatsApp group. It calculates the total number of participants, total messages, and message contribution of each member. It also determines the chat duration using the first and last message dates.

In [4]:
group_name = "Hostel Bois 4ever"
period = "01 April 2024 to 30 May 2024 (60 days)"
participants = []
message_count = {}

# Count messages of each participant
for msg in messages:

    sender = msg["sender"]

    if sender not in participants:
        participants.append(sender)

    if sender in message_count:
        message_count[sender] = message_count[sender] + 1
    else:
        message_count[sender] = 1


# Print Output

print("=" * 60)
print("GROUP OVERVIEW")
print("=" * 60)

print("Group            :", group_name)
print("Period           :", period)
print("Total Messages   :", len(messages))
print("Participants     :", len(participants))

print()
print("MESSAGES PER PERSON")
print()


for name in participants:

    total = message_count[name]

    percentage = (total / len(messages)) * 100

    print(f"{name:<15}: {total:>4} ({percentage:.1f}%)")

GROUP OVERVIEW
Group            : Hostel Bois 4ever
Period           : 01 April 2024 to 30 May 2024 (60 days)
Total Messages   : 3174
Participants     : 6

MESSAGES PER PERSON

Rahul          :  953 (30.0%)
Priya          :  718 (22.6%)
Karan          :  354 (11.2%)
Neha           :  635 (20.0%)
Aman           :  490 (15.4%)
Vikas          :   24 (0.8%)


Feature 3 – Most Active Day & Hour

This feature analyzes the chat activity based on date and time. It counts messages for every day and every hour to find the busiest period. The results help identify when the group is most active.

In [5]:
day_count = {}
hour_count = {}


for msg in messages:


    date = msg["date"]


    hour = msg["time"].split(":")[0]

    # Count day wise messages
    if date in day_count:
        day_count[date] = day_count[date] + 1
    else:
        day_count[date] = 1

    # Count hour wise messages
    if hour in hour_count:
        hour_count[hour] = hour_count[hour] + 1
    else:
        hour_count[hour] = 1


# Find busiest day

busiest_day = ""
max_day_messages = 0

for date in day_count:

    if day_count[date] > max_day_messages:

        max_day_messages = day_count[date]
        busiest_day = date


# Find busiest hour

busiest_hour = ""
max_hour_messages = 0

for hour in hour_count:

    if hour_count[hour] > max_hour_messages:

        max_hour_messages = hour_count[hour]
        busiest_hour = hour


# Calculate average messages per day

total_days = len(day_count)

average_messages = round(max_hour_messages / total_days)


next_hour = str((int(busiest_hour) + 1) % 24).zfill(2)
print("=" * 60)
print("MOST ACTIVE DAY AND HOUR")
print("=" * 60)

print()

print("Busiest day  :", busiest_day,
      "(" + str(max_day_messages) + " messages)")

print("Busiest hour :", busiest_hour + ":00 - " + next_hour + ":00",
      "(avg " + str(average_messages) + " messages per day)")

MOST ACTIVE DAY AND HOUR

Busiest day  : 04/05/24 (76 messages)
Busiest hour : 18:00 - 19:00 (avg 4 messages per day)


Feature 4 – Activity Heatmap

This feature creates a NumPy matrix to represent each participant's activity across all 24 hours. Every message updates the corresponding row and hour in the matrix. The matrix is displayed as a text-based heatmap using different symbols.

In [6]:
participants = []

for msg in messages:

    if msg["sender"] not in participants:
        participants.append(msg["sender"])

# Create a 6 x 24 matrix

heatmap = np.zeros((len(participants), 24), dtype=int)

# Fill the matrix

for msg in messages:

    sender = msg["sender"]

    hour = int(msg["time"].split(":")[0])

    row = participants.index(sender)

    heatmap[row][hour] = heatmap[row][hour] + 1


print("=" * 65)
print("ACTIVITY HEATMAP (messages by hour)")
print("=" * 65)

# Print hour headings

print(" " * 12, end="")

for hour in range(0, 24, 3):
    print(f"{hour:02}", end="  ")

print()

# Print heatmap

for i in range(len(participants)):

    print(f"{participants[i]:<10}", end=" ")

    max_value = np.max(heatmap[i])

    for hour in range(0, 24, 3):

        value = heatmap[i][hour]

        if max_value == 0:
            symbol = "."

        else:

            ratio = value / max_value

            if ratio == 0:
                symbol = "."
            elif ratio <= 0.25:
                symbol = "░"
            elif ratio <= 0.50:
                symbol = "▒"
            else:
                symbol = "█"

        print(symbol, end="   ")

    print()

ACTIVITY HEATMAP (messages by hour)
            00  03  06  09  12  15  18  21  
Rahul      ░   ░   ░   ░   █   █   █   █   
Priya      .   .   ░   █   █   ▒   █   ▒   
Karan      .   .   .   ▒   █   █   █   ▒   
Neha       .   .   ░   █   █   ░   █   ▒   
Aman       █   █   .   .   .   ░   ░   ░   
Vikas      .   .   .   ▒   █   ▒   █   ▒   


Feature 5 – Top Words

This feature finds the most frequently used words in the chat. It converts all words to lowercase, removes punctuation, and ignores common stop words. The remaining words are counted and displayed with a simple bar representation.

In [7]:
word_count = {}

# Stop words
stop_words = [
    "i","me","my","mine","you","your","yours",
    "he","his","him","she","her","hers",
    "it","its","we","our","ours","they","their",
    "the","a","an","is","am","are","was","were","be","been","being",
    "and","or","but","if","then","so","because","as",
    "to","of","in","on","at","for","from","with","by","about","into",
    "this","that","these","those","there","here",
    "have","has","had","do","does","did",
    "can","could","will","would","should","may","might","must",
    "how","what","when","where","which","who","why",
    "just","today","tomorrow","yesterday",
    "everyone","telling","up",
    "ok","okay","yes","no",
    "hai","haan","nah","lol"
]

# Punctuation characters
punctuation = ".,!?;:'\"()[]{}<>/@#$%^&*-_=+`~"

# Read every message
for msg in messages:

    message = msg["message"].lower()

    # Skip media and deleted messages
    if "<media omitted>" in message:
        continue

    if "deleted this message" in message:
        continue

    words = message.split()

    for word in words:


        word = word.strip(punctuation)


        if word == "":
            continue


        if word.isdigit():
            continue


        if len(word) < 3:
            continue


        if word in stop_words:
            continue

        # Count words
        if word in word_count:
            word_count[word] = word_count[word] + 1
        else:
            word_count[word] = 1


# Sort words by frequency
sorted_words = sorted(word_count.items(),
                      key=lambda item: item[1],
                      reverse=True)

# Maximum count for scaling bars
max_count = sorted_words[0][1]

print("=" * 60)
print("THIS GROUP'S FAVOURITE WORDS")
print("=" * 60)
print()

# Print Top 10 words
for word, count in sorted_words[:10]:

    # Scale bar to maximum length of 30 blocks
    bar_length = int((count / max_count) * 30)

    if bar_length == 0:
        bar_length = 1

    bar = "█" * bar_length

    print(f"{word:<15} {bar:<30} {count}")

THIS GROUP'S FAVOURITE WORDS

guys            ██████████████████████████████ 318
bhai            ███████████████                160
one             ██████████████                 157
started         ██████████████                 150
scene           █████████████                  145
entire          █████████████                  145
please          █████████████                  141
anyone          █████████████                  139
yaar            █████████████                  139
kya             ████████████                   133


Feature 6 – Response Speed & Silent Streaks

This feature calculates the average response time of each participant using the datetime module. It also finds the longest period during which a participant remained inactive. These statistics help identify active and inactive members.

In [8]:
from datetime import datetime


response_time = {}

for person in participants:
    response_time[person] = []

for i in range(1, len(messages)):

    previous = messages[i - 1]
    current = messages[i]


    if previous["sender"] != current["sender"]:

        time1 = datetime.strptime(
            previous["date"] + ", " + previous["time"],
            "%d/%m/%y, %H:%M"
        )

        time2 = datetime.strptime(
            current["date"] + ", " + current["time"],
            "%d/%m/%y, %H:%M"
        )

        gap = (time2 - time1).total_seconds()

        if gap >= 0:
            response_time[current["sender"]].append(gap)



average = {}

for person in participants:

    if len(response_time[person]) > 0:

        average[person] = sum(response_time[person]) / len(response_time[person])

    else:

        average[person] = 999999999


fastest = min(average, key=average.get)
slowest = max(average, key=average.get)


all_dates = []

for msg in messages:

    if msg["date"] not in all_dates:
        all_dates.append(msg["date"])


silent_days = {}

for person in participants:

    longest = 0
    current = 0

    for date in all_dates:

        active = False

        for msg in messages:

            if msg["sender"] == person and msg["date"] == date:
                active = True
                break

        if active:

            current = 0

        else:

            current += 1

            if current > longest:
                longest = current

    silent_days[person] = longest
print("=" * 60)
print("RESPONSE PATTERNS")
print("=" * 60)

if average[fastest] < 3600:

    print("Fastest Replier :", fastest,
          f"(avg {average[fastest]/60:.1f} minutes)")

else:

    print("Fastest Replier :", fastest,
          f"(avg {average[fastest]/3600:.1f} hours)")



if average[slowest] < 3600:

    print("Slowest Replier :", slowest,
          f"(avg {average[slowest]/60:.1f} minutes)")

else:

    print("Slowest Replier :", slowest,
          f"(avg {average[slowest]/3600:.1f} hours)")


print()
print("LONGEST SILENT STREAKS")
print("-" * 60)

for person in participants:

    if silent_days[person] == 0:

        print(f"{person:<10} : 0 days (never went silent)")

    else:

        print(f"{person:<10} : {silent_days[person]} days")

RESPONSE PATTERNS
Fastest Replier : Rahul (avg 34.9 minutes)
Slowest Replier : Aman (avg 55.4 minutes)

LONGEST SILENT STREAKS
------------------------------------------------------------
Rahul      : 0 days (never went silent)
Priya      : 0 days (never went silent)
Karan      : 0 days (never went silent)
Neha       : 0 days (never went silent)
Aman       : 0 days (never went silent)
Vikas      : 11 days


Feature 7 – Personality Archetype Detection

This feature assigns a personality type to every participant based on their chat behaviour. It uses factors like message count, response speed, night activity, and silent streaks. Each member is given the most suitable archetype according to predefined rules.

In [9]:
print("=" * 60)
print("PERSONALITY ARCHETYPES")
print("=" * 60)
print()

for person in participants:

    total = message_count[person]

    night_messages = 0
    capital_messages = 0
    total_words = 0

    for msg in messages:

        if msg["sender"] == person:

            text = msg["message"]

            # Count words
            total_words += len(text.split())

            # Night messages
            hour = int(msg["time"].split(":")[0])

            if hour >= 23 or hour <= 4:
                night_messages += 1

            # ALL CAPS messages
            if text.isupper() and len(text) > 3:
                capital_messages += 1

    # Calculations
    avg_words = total_words / total if total > 0 else 0
    night_percent = (night_messages / total) * 100 if total > 0 else 0
    caps_percent = (capital_messages / total) * 100 if total > 0 else 0

    # Decide Archetype

    if person == max(message_count, key=message_count.get):

        reason = f"{total} messages"
        archetype = "THE SPAMMER"

    elif person == min(average, key=average.get):

        if average[person] < 3600:
            reason = f"avg {average[person]/60:.1f} minutes"
        else:
            reason = f"avg {average[person]/3600:.1f} hours"

        archetype = "THE GROUP MOM"

    elif night_percent >= 50:

        reason = f"{night_percent:.1f}% msgs after 11 PM"
        archetype = "THE NIGHT OWL"

    elif avg_words >= 15:

        reason = f"avg {avg_words:.1f} words/msg"
        archetype = "THE STORYTELLER"

    elif caps_percent >= 20:

        reason = f"{caps_percent:.1f}% ALL CAPS msgs"
        archetype = "THE DRAMA QUEEN"

    elif person == max(silent_days, key=silent_days.get):

        reason = f"silent {silent_days[person]} days"
        archetype = "THE GHOST"

    else:

        reason = "balanced activity"
        archetype = "THE ACTIVE MEMBER"

    print(f"{person:<10} -> {archetype:<18} ({reason})")

PERSONALITY ARCHETYPES

Rahul      -> THE SPAMMER        (953 messages)
Priya      -> THE ACTIVE MEMBER  (balanced activity)
Karan      -> THE STORYTELLER    (avg 55.7 words/msg)
Neha       -> THE DRAMA QUEEN    (62.2% ALL CAPS msgs)
Aman       -> THE NIGHT OWL      (79.8% msgs after 11 PM)
Vikas      -> THE GHOST          (silent 11 days)


Feature 8 – Final Report

This feature combines all the analysis results into one organized report. It displays the group overview, activity summary, top words, response patterns, and personality archetypes. The report is formatted clearly for easy understanding and presentation.

These are concise, natural, and different enough that your friend's notebook won't look like a copy while still explaining the same features.

In [10]:
print("=" * 65)
print('          GROUPDNA REPORT - "Hostel Bois 4ever"')
print(f'      {len(day_count)} days  •  {len(messages)} messages  •  {len(participants)} members')
print("=" * 65)

# ------------------------------------------------------------
# Period
# ------------------------------------------------------------

print()

from datetime import datetime

dates = sorted(
    day_count.keys(),
    key=lambda x: datetime.strptime(x, "%d/%m/%y")
)

start_date = dates[0]
end_date = dates[-1]

print(f'{"Period":<18}: {start_date} to {end_date}')
print(f'{"Busiest Day":<18}: {busiest_day}')
print(f'{"Busiest Hour":<18}: {busiest_hour}')

# ------------------------------------------------------------
# Messages Per Person
# ------------------------------------------------------------

print("\nMESSAGES PER PERSON\n")

highest = max(message_count.values())

for person in participants:

    count = message_count[person]

    percent = (count / len(messages)) * 100

    bars = "█" * int((count / highest) * 20)

    print(f'{person:<10} {bars:<20} {count:>4} ({percent:.1f}%)')

# ------------------------------------------------------------
# Activity Heatmap
# ------------------------------------------------------------

print("\nACTIVITY HEATMAP (hour of day)\n")

hours = [0,3,6,9,12,15,18,21]

print("       ", end="")

for h in hours:
    print(f"{h:02}", end="  ")

print()

symbols = [".","░","▒","█"]

for i, person in enumerate(participants):

    print(f"{person:<8}", end="")

    row = heatmap[i]

    row_max = max(row)

    if row_max == 0:
        row_max = 1

    for h in hours:

        value = row[h]

        ratio = value / row_max

        if ratio == 0:
            s = "."
        elif ratio <= 0.25:
            s = "░"
        elif ratio <= 0.75:
            s = "▒"
        else:
            s = "█"

        print(f" {s} ", end="")

    print()

# ------------------------------------------------------------
# Top Words
# ------------------------------------------------------------

print("\nTHIS GROUP'S FAVOURITE WORDS\n")

highest_word = sorted_words[0][1]

for word, count in sorted_words[:5]:

    bars = "█" * int((count / highest_word) * 20)

    print(f'{word:<10} {bars:<20} {count}')

# ------------------------------------------------------------
# Response Pattern
# ------------------------------------------------------------

print("\nRESPONSE PATTERNS\n")

if average[fastest] < 3600:
    fast = f"{average[fastest]/60:.1f} minutes"
else:
    fast = f"{average[fastest]/3600:.1f} hours"

if average[slowest] < 3600:
    slow = f"{average[slowest]/60:.1f} minutes"
else:
    slow = f"{average[slowest]/3600:.1f} hours"

print(f"Fastest replier : {fastest} (avg {fast})")
print(f"Slowest replier : {slowest} (avg {slow})")

# ------------------------------------------------------------
# Silent Streak
# ------------------------------------------------------------

print("\nLONGEST SILENT STREAKS\n")

for person in participants:

    if silent_days[person] == 0:
        print(f"{person:<10}: Never went silent")

    else:
        print(f"{person:<10}: {silent_days[person]} days")

# ------------------------------------------------------------
# Personality
# ------------------------------------------------------------

print("\nPERSONALITY ARCHETYPES\n")

for person in participants:

    total = message_count[person]

    words = 0
    caps = 0
    night = 0

    for msg in messages:

        if msg["sender"] == person:

            words += len(msg["message"].split())

            hour = int(msg["time"].split(":")[0])

            if hour >= 23 or hour <= 4:
                night += 1

            if msg["message"].isupper() and len(msg["message"]) > 3:
                caps += 1

    avg_words = words / total
    night_percent = (night / total) * 100
    caps_percent = (caps / total) * 100

    if person == max(message_count, key=message_count.get):
        role = "THE SPAMMER"

    elif person == min(average, key=average.get):
        role = "THE GROUP MOM"

    elif night_percent >= 50:
        role = "THE NIGHT OWL"

    elif avg_words >= 15:
        role = "THE STORYTELLER"

    elif caps_percent >= 20:
        role = "THE DRAMA QUEEN"

    elif person == max(silent_days, key=silent_days.get):
        role = "THE GHOST"

    else:
        role = "THE ACTIVE MEMBER"

    print(f"{person:<10} -> {role}")

print("\n" + "=" * 65)
print("               END OF REPORT")
print("=" * 65)

          GROUPDNA REPORT - "Hostel Bois 4ever"
      60 days  •  3174 messages  •  6 members

Period            : 01/04/24 to 30/05/24
Busiest Day       : 04/05/24
Busiest Hour      : 18

MESSAGES PER PERSON

Rahul      ████████████████████  953 (30.0%)
Priya      ███████████████       718 (22.6%)
Karan      ███████               354 (11.2%)
Neha       █████████████         635 (20.0%)
Aman       ██████████            490 (15.4%)
Vikas                             24 (0.8%)

ACTIVITY HEATMAP (hour of day)

       00  03  06  09  12  15  18  21  
Rahul    ░  ░  ░  ░  ▒  ▒  █  █ 
Priya    .  .  ░  █  █  ▒  ▒  ▒ 
Karan    .  .  .  ▒  █  ▒  ▒  ▒ 
Neha     .  .  ░  █  ▒  ░  █  ▒ 
Aman     ▒  ▒  .  .  .  ░  ░  ░ 
Vikas    .  .  .  ▒  ▒  ▒  ▒  ▒ 

THIS GROUP'S FAVOURITE WORDS

guys       ████████████████████ 318
bhai       ██████████           160
one        █████████            157
started    █████████            150
scene      █████████            145

RESPONSE PATTERNS

Fastest replier : R